Notebook Purpose: test the original model (NO WRAPPER) with MBDS

Uses various types of windowed data and calibration settings

### Setup - Load Imports + x, y train and test data

In [27]:
# Imports
import os
import json

import sys
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ProjectedGradientDescent
from art.attacks.evasion import FastGradientMethod
from art.attacks.evasion import SaliencyMapMethod
from art.attacks.evasion import CarliniL2Method
# from art.attacks.evasion import Carlin
from sklearn.preprocessing import MinMaxScaler

sys.path.append(str(Path.cwd().parents[0]))

import os

import torch
from utils.models import OBU
from utils.functions import get_windowed_data, load_model_checkpoint

from sklearn.metrics import precision_score, recall_score, f1_score

import argparse
import json
import sys
import os
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from typing import Dict, Any

import pandas as pd

from MBD_systems.data_structures import Parameters
import MBD_systems.data_processing

In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
# Set Inputs
# checkpoint_file="../../../saved_models/RandomPos-cenFL-updated.ckpt"
data_file = "../data/RandomPos_0709.csv"
train_perc = 80

norm_trained = False
collapse_output = True # True if using JSMA (maybe CW????)


In [29]:
# Loading training and testing data
from utils.functions import get_windowed_data
(x_train, y_train), (x_test, y_test), fed_dataset = get_windowed_data(data_file, 
                                                                      normalize=norm_trained, 
                                                                      train_perc=train_perc)
len(x_test), len(y_test)

(124774, 124774)

### Test MBD_systems

In [25]:
# Evaluated senderID windowed inputs with MBDS 
import time
from MBD_systems.windowed_eval import evaluate_windowed_ids
t0 = time.time()
metrics = evaluate_windowed_ids('../data/RandomPos_0709.csv', attacker_code=3, workers=20)

print(f'elapsed: {time.time()-t0:.1f}s')

Splitting ../data/RandomPos_0709.csv by SenderID into ../data/_sender_split_RandomPos_0709 ...
Found 4067 senders, evaluating windows with 20 workers...
Processed 500/4067 senders
Processed 1000/4067 senders
Processed 1500/4067 senders
Processed 2000/4067 senders
Processed 2500/4067 senders
Processed 3000/4067 senders
Processed 3500/4067 senders
Processed 4000/4067 senders
Processed 4067/4067 senders


elapsed: 46.3s


In [ ]:
# JSON Output
print(json.dumps(metrics, indent=1))

{
 "tp": 184237,
 "tn": 9254,
 "fp": 430094,
 "fn": 281,
 "accuracy": 0.31014833313564133,
 "precision": 0.2998985888714716,
 "recall": 0.9984771133439556,
 "check_counts": {
  "range_plausibility": 1488846,
  "speed_plausibility": 1789074,
  "sudden_appearance": 274103,
  "position_consistency": 5181927,
  "speed_consistency": 688932,
  "position_speed_consistency": 5244656
 },
 "f1": 0.46125613226028955
}


In [ ]:
# Evaluate senderID and receiverID windowed inputs with MBDS
from MBD_systems.windowed_eval import evaluate_windowed_ids_by_pair
metrics = evaluate_windowed_ids_by_pair('../data/RandomPos_0709.csv', attacker_code=3, workers=20)

Splitting ../data/RandomPos_0709.csv by (SenderID, RecieverID) into ../data/_pair_split_RandomPos_0709 ...
Found 147435 sender/receiver pairs, evaluating windows with 20 workers...
Processed 5000/147435 pairs
Processed 10000/147435 pairs
Processed 15000/147435 pairs
Processed 20000/147435 pairs
Processed 25000/147435 pairs
Processed 30000/147435 pairs
Processed 35000/147435 pairs
Processed 40000/147435 pairs
Processed 45000/147435 pairs
Processed 50000/147435 pairs
Processed 55000/147435 pairs
Processed 60000/147435 pairs
Processed 65000/147435 pairs
Processed 70000/147435 pairs
Processed 75000/147435 pairs
Processed 80000/147435 pairs
Processed 85000/147435 pairs
Processed 90000/147435 pairs
Processed 95000/147435 pairs
Processed 100000/147435 pairs
Processed 105000/147435 pairs
Processed 110000/147435 pairs
Processed 115000/147435 pairs
Processed 120000/147435 pairs
Processed 125000/147435 pairs
Processed 130000/147435 pairs
Processed 135000/147435 pairs
Processed 140000/147435 pairs

In [ ]:
# Add calibration to windowed ID pairs
from MBD_systems.calibrate_thresholds import calibrate_parameters

calibration = calibrate_parameters('../data/RandomPos_0709.csv', train_perc=80, percentile=99, workers=20)
print(calibration['summary'])
print(calibration['params'])


Scanning ../data/RandomPos_0709.csv for rcvTime bounds...
Train/test cutoff rcvTime=30961.75 (train_perc=80%)
Found 147435 pairs, collecting benign-only train-slice stats with 20 workers...
Processed 5000/147435 pairs
Processed 10000/147435 pairs
Processed 15000/147435 pairs
Processed 20000/147435 pairs
Processed 25000/147435 pairs
Processed 30000/147435 pairs
Processed 35000/147435 pairs
Processed 40000/147435 pairs
Processed 45000/147435 pairs
Processed 50000/147435 pairs
Processed 55000/147435 pairs
Processed 60000/147435 pairs
Processed 65000/147435 pairs
Processed 70000/147435 pairs
Processed 75000/147435 pairs
Processed 80000/147435 pairs
Processed 85000/147435 pairs
Processed 90000/147435 pairs
Processed 95000/147435 pairs
Processed 100000/147435 pairs
Processed 105000/147435 pairs
Processed 110000/147435 pairs
Processed 115000/147435 pairs
Processed 120000/147435 pairs
Processed 125000/147435 pairs
Processed 130000/147435 pairs
Processed 135000/147435 pairs
Processed 140000/147

{'distances': {'n': 1700158, 'p99': 387.1118633926308}, 'speeds': {'n': 1700158, 'p99': 76.91397405173522}, 'implied_speeds': {'n': 1511740, 'p99': 29.70016790986721}, 'implied_accels': {'n': 861127, 'p99': 14.436968025583893}, 'implied_decels': {'n': 650613, 'p99': 120.51659579662054}, 'mahalanobis': {'n': 1511740, 'p99': 6724.971445148739}, 'first_distance': {'n': 79067, 'p99': 425.50069702360787}}
Parameters(MAX_PLAUSIBLE_RANGE=387.1118633926308, MAX_SA_RANGE=12.788586168961647, MAX_PLAUSIBLE_DIST_NEGATIVE=-4.498043609151968, MAX_PLAUSIBLE_SPEED=76.91397405173522, MAX_PLAUSIBLE_ACCEL=14.436968025583893, MAX_PLAUSIBLE_DECEL=120.51659579662054, MAX_HEADING_CHANGE=76.23960198184574, MAX_DELTA_INTERSECTION=4.297205392054125, MAX_TIME_DELTA=4.7243132992341765, POS_HEADING_TIME=0.39809268515981683, MAX_MGT_RNG_UP=0.7063553998663573, MAX_MGT_RNG_DOWN=1.0299180161460522, MAX_SA_TIME=2.1, MAX_NON_ROUTE_SPEED=0.6798715937399384)


In [ ]:
# Run with windowed pairs and calibration
from MBD_systems.windowed_eval import evaluate_windowed_ids_by_pair

metrics_calibrated = evaluate_windowed_ids_by_pair(
    '../data/RandomPos_0709.csv',
    attacker_code=3,
    workers=20,
    params=calibration['params']
)
print(metrics_calibrated)


Splitting ../data/RandomPos_0709.csv by (SenderID, RecieverID) into ../data/_pair_split_RandomPos_0709 ...
Found 147435 sender/receiver pairs, evaluating windows with 20 workers...
Processed 5000/147435 pairs
Processed 10000/147435 pairs
Processed 15000/147435 pairs
Processed 20000/147435 pairs
Processed 25000/147435 pairs
Processed 30000/147435 pairs
Processed 35000/147435 pairs
Processed 40000/147435 pairs
Processed 45000/147435 pairs
Processed 50000/147435 pairs
Processed 55000/147435 pairs
Processed 60000/147435 pairs
Processed 65000/147435 pairs
Processed 70000/147435 pairs
Processed 75000/147435 pairs
Processed 80000/147435 pairs
Processed 85000/147435 pairs
Processed 90000/147435 pairs
Processed 95000/147435 pairs
Processed 100000/147435 pairs
Processed 105000/147435 pairs
Processed 110000/147435 pairs
Processed 115000/147435 pairs
Processed 120000/147435 pairs
Processed 125000/147435 pairs
Processed 130000/147435 pairs
Processed 135000/147435 pairs
Processed 140000/147435 pairs

{'tp': 126777, 'tn': 44385, 'fp': 257820, 'fn': 313, 'accuracy': 0.3987048533060017, 'precision': 0.3296359565987254, 'recall': 0.9975371783775278, 'check_counts': {'range_plausibility': 1071854, 'speed_plausibility': 1238063, 'sudden_appearance': 23637, 'position_consistency': 1018645, 'speed_consistency': 942415, 'position_speed_consistency': 2819128}, 'f1': 0.4955255849767533}


In [ ]:
# Look at metrics
metrics

{'tp': 184237,
 'tn': 9254,
 'fp': 430094,
 'fn': 281,
 'accuracy': 0.31014833313564133,
 'precision': 0.2998985888714716,
 'recall': 0.9984771133439556,
 'check_counts': {'range_plausibility': 1488846,
  'speed_plausibility': 1789074,
  'sudden_appearance': 274103,
  'position_consistency': 5181927,
  'speed_consistency': 688932,
  'position_speed_consistency': 5244656},
 'f1': 0.46125613226028955}

### Running MBDS like in VeReMi-Extension

In [ ]:
!python "../MBD_systems/main.py" --input_folder="../data/TRAIN/" --type=0

Starting processing with 24 workers...
0
